In [1]:
# Install dependencies
%pip install anthropic python-dotenv
%pip install dotenv
%pip install IPython


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Create an API Client
from anthropic import Anthropic

client=Anthropic()
model="claude-haiku-4-5"

In [4]:
# Define helper functions to support multi-message converstation
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text    

In [5]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()
print(dataset)

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a Python function that validates if a given string is a valid AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, must start and end with a letter or number).'}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 8 or 17 hexadecimal characters).'}]


In [21]:
from statistics import mean

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Respond with a JSON object containing exactly these keys:
    "strengths": an array of 1-3 strings
    "weaknesses": an array of 1-3 strings
    "reasoning": a string
    "score": a number between 1 and 10
    
    Do not include any other text or explanation.
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    
    eval_text = chat(messages)
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        return {
            "strengths": ["Unable to evaluate"],
            "weaknesses": ["Response not in JSON format"],
            "reasoning": f"Model response: {eval_text[:200]}...",
            "score": 0
        }

def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade.get("score", 0)  # Default to 0 if missing
    reasoning = model_grade.get("reasoning", "No reasoning provided")
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 0
[
  {
    "output": "# AWS S3 Bucket Name Validator\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a given string is a valid AWS S3 bucket name.\n    \n    AWS S3 Bucket Naming Rules:\n    - Length: 3-63 characters\n    - Characters: lowercase letters, numbers, and hyphens only\n    - Must start with a letter or number\n    - Must end with a letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address\n    \n    Args:\n        bucket_name (str): The bucket name to validate\n        \n    Returns:\n        bool: True if valid, False otherwise\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    # Check if it contains only lowercase letters, numbers, and hyphens\n    i